# Joint outcomes under binary interventions

We extend the paper's binary `causl` setting to two outcomes. Under `do(A=a)`, the true joint law is Gaussian with mean `(2a, -a)`, unit variances and covariance `0.4`. We compare Frengression with joint conditional engression plus g-computation, and with a shuffled-pair diagnostic.

**Restart Kernel → Run All:** 1000 training updates, `noise_dim=2`, 5000 outcome pairs per arm, and 30 paired replications. Completed replications are reused. Results and figures are saved to `~/frengression_results/joint_outcomes_causl/`.

Requires R with `causl`, plus `torch`, `engression`, `numpy`, `pandas`, `matplotlib` and `scipy`. To redraw completed figures without refitting, run `python examples/replot_joint_outcomes.py` from the repository folder.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import json
import hashlib
import platform
from functools import lru_cache
from importlib import metadata as package_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.integrate import quad
from scipy.stats import norm
from engression.models import StoNet
from engression.loss_func import energy_loss_two_sample

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'frengression').is_dir()
             and (p / 'examples' / 'causl_joint_outcomes.R').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the frengression_pkg repository or examples/ folder.')
sys.path.insert(0, str(ROOT / 'src'))
from frengression import Frengression
sys.path.insert(0, str(ROOT / 'examples'))
from joint_outcome_reporting import (atomic_csv, save_draw_checkpoint, load_draw_checkpoint,
                                    plot_joint_distributions, plot_paired_errors)

RSCRIPT = shutil.which('Rscript')
if RSCRIPT is None:
    raise RuntimeError('Rscript was not found. Install R and the R package causl.')
check = subprocess.run([RSCRIPT, '-e', 'stopifnot(requireNamespace("causl", quietly=TRUE))'],
                       capture_output=True, text=True)
if check.returncode:
    raise RuntimeError('R package causl is missing. In R: remotes::install_github("rje42/causl")'
                       + '\n' + check.stderr[-1000:])
print('Repository:', ROOT)
print('Rscript:', RSCRIPT)
print('torch:', torch.__version__)

## Generate the observational data

Five instruments and five confounders follow the paper's binary setting. Causal truth is specified by the R generator's parameters.


In [ ]:
COVARIATES = [f'I{i}' for i in range(1, 6)] + [f'X{i}' for i in range(1, 6)]
TRUE_COV = np.array([[1.0, 0.4], [0.4, 1.0]])
TRUE_MEAN = {0: np.array([0.0, 0.0]), 1: np.array([2.0, -1.0])}

def generate_causl_joint(n=5000, seed=2026, strength_instr=1.0, randomized=False):
    with tempfile.TemporaryDirectory(prefix='causl_joint_') as temp:
        csv_path = Path(temp) / 'observed.csv'
        cmd = [RSCRIPT, str(ROOT / 'examples' / 'causl_joint_outcomes.R'),
               str(csv_path), str(n), str(seed), str(strength_instr)]
        if randomized:
            cmd.append('randomized')
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode:
            raise RuntimeError('Native causl generation failed:\n' +
                               run.stderr[-4000:] + '\n' + run.stdout[-1000:])
        frame = pd.read_csv(csv_path)
    expected = COVARIATES + ['A', 'Y1', 'Y2']
    if frame.columns.tolist() != expected or len(frame) != n:
        raise ValueError(f'Unexpected causl columns/rows: {frame.shape}, {frame.columns.tolist()}')
    x = frame[['A']].to_numpy(dtype=np.float32)
    z = frame[COVARIATES].to_numpy(dtype=np.float32)
    y = frame[['Y1', 'Y2']].to_numpy(dtype=np.float32)
    if not np.isfinite(np.concatenate([z, x, y], axis=1)).all() or not np.isin(x, [0, 1]).all():
        raise ValueError('causl produced non-finite values or treatment outside {0,1}')
    if min((x[:, 0] == 0).sum(), (x[:, 0] == 1).sum()) == 0:
        raise ValueError('One observed treatment arm is empty')
    # This sanity check uses the observational residuals, never a simulated truth set.
    residual = y - np.column_stack((2 * x[:, 0], -x[:, 0]))
    empirical_cov = np.cov(residual, rowvar=False)
    if n >= 3000 and abs(empirical_cov[0, 1] - TRUE_COV[0, 1]) > 0.15:
        raise RuntimeError('causl residual covariance does not match the specified 0.4. '
                           f'Observed={empirical_cov[0,1]:.3f}; inspect the R/copula version before fitting.')
    return x, z, y, empirical_cov

N, SEED, STRENGTH_INSTR = 5000, 2026, 1.0
x, z, y, observed_residual_cov = generate_causl_joint(N, SEED, STRENGTH_INSTR)
print('x, z, y shapes:', x.shape, z.shape, y.shape)
print('Observed residual cross-covariance (sanity check):', round(observed_residual_cov[0,1], 3))
print('Specified causal cross-covariance:', TRUE_COV[0,1])


### Check the specified causal law

A separate randomized-treatment sample checks the generator against the specified moments. It is used for diagnostics, not to define ground truth.


In [ ]:
x_check, _, y_check, _ = generate_causl_joint(
    n=20000, seed=SEED + 1, strength_instr=STRENGTH_INSTR, randomized=True)
check_rows = []
for arm in (0, 1):
    d = y_check[x_check[:, 0] == arm]
    c = np.cov(d, rowvar=False)
    check_rows.append(dict(arm=arm, count=len(d),
                           mean1=d[:, 0].mean(), expected_mean1=TRUE_MEAN[arm][0],
                           mean2=d[:, 1].mean(), expected_mean2=TRUE_MEAN[arm][1],
                           var1=c[0, 0], var2=c[1, 1],
                           cov12=c[0, 1], expected_cov12=TRUE_COV[0, 1]))
check_table = pd.DataFrame(check_rows)
display(check_table.round(3))
print('Diagnostic only: the exact values in the expected columns define the truth.')


## Fit the joint Frengression model

A single network generates both outcomes together. We use three hidden layers of width 100, `noise_dim=2`, learning rate `1e-4`, and 1000 updates. Outcome pairs are retained when converting the sampling array.


In [ ]:
FR_ITERS = 1000
MC_DRAWS = 5000
torch.set_num_threads(1)

def fit_joint(x, z, y, seed, iterations=None, draws=None):
    iterations = FR_ITERS if iterations is None else iterations
    draws = MC_DRAWS if draws is None else draws
    torch.manual_seed(seed)
    model = Frengression(x_dim=1, y_dim=2, z_dim=10,
                         num_layer=3, hidden_dim=100, noise_dim=2,
                         x_binary=True, z_binary_dims=0, device=torch.device('cpu'))
    xt, zt, yt = (torch.as_tensor(v, dtype=torch.float32) for v in (x, z, y))
    start = time.perf_counter()
    model.train_y(xt, zt, yt, num_iters=iterations, lr=1e-4,
                  print_every_iter=max(1, iterations // 5), tol=0.0)
    seconds = time.perf_counter() - start
    arm_draws = {}
    for arm in (0, 1):
        sample = model.sample_causal_margin(torch.tensor([arm], dtype=torch.int32),
                                            sample_size=draws)
        # StoNet.sample returns [data_size, response_dim, sample_size].
        # Direct reshape(draws, 2) would scramble the two outcomes.
        if tuple(sample.shape) != (1, y.shape[1], draws):
            raise RuntimeError(f'Unexpected joint sample shape: {tuple(sample.shape)}')
        arm_draws[arm] = sample[0].transpose(0, 1).contiguous().cpu().numpy()
        if not np.isfinite(arm_draws[arm]).all():
            raise ValueError(f'Non-finite Frengression sample for arm {arm}')
    return model, arm_draws, seconds

joint_model, joint_draws, fit_seconds = fit_joint(x, z, y, SEED)
print(f'One joint fit: {fit_seconds:.1f} seconds')
print('Frengression noise_dim=2; joint samples transposed to [draws, outcomes]')

## Moments and shuffled pairs

Within each treatment arm, **shuffled pairs** keep `Y1` in place and randomly permute `Y2` across generated rows:
`(Y1[i], Y2[i]) → (Y1[i], Y2[permutation[i]])`.
Both empirical outcome margins remain exactly unchanged; the original pairing is broken. This is a dependence diagnostic, not another fitted method.


In [ ]:
def evaluate_joint(draws, label):
    rows = []
    for arm in (0, 1):
        d = np.asarray(draws[arm], dtype=float)
        mu, cov = d.mean(axis=0), np.cov(d, rowvar=False)
        rows.append(dict(method=label, arm=arm, true_mean1=TRUE_MEAN[arm][0],
                         estimated_mean1=mu[0], true_mean2=TRUE_MEAN[arm][1],
                         estimated_mean2=mu[1], mean_mae=np.abs(mu - TRUE_MEAN[arm]).mean(),
                         true_var1=1.0, estimated_var1=cov[0,0],
                         true_var2=1.0, estimated_var2=cov[1,1],
                         true_cov12=0.4, estimated_cov12=cov[0,1],
                         cov12_error=abs(cov[0,1] - 0.4),
                         covariance_fro_error=np.linalg.norm(cov - TRUE_COV, ord='fro')))
    return pd.DataFrame(rows)

rng = np.random.default_rng(SEED + 99)
shuffled = {arm: np.column_stack((joint_draws[arm][:, 0],
             rng.permutation(joint_draws[arm][:, 1]))) for arm in (0, 1)}
pilot_table = pd.concat([evaluate_joint(joint_draws, 'frengression_joint'),
                         evaluate_joint(shuffled, 'shuffled_diagnostic')], ignore_index=True)
display(pilot_table.round(4))


### Optional longer fit

The paper uses 1000 updates. The separate 4000-update sensitivity check is off by default.


In [ ]:
LONGER_ITERS = 4000
RUN_LONGER_CHECK = False
PAPER_FIT = 'pilot'
if RUN_LONGER_CHECK or PAPER_FIT == 'longer':
    longer_model, longer_draws, longer_seconds = fit_joint(
        x, z, y, SEED, iterations=LONGER_ITERS)
    longer_table = evaluate_joint(longer_draws, f'frengression_{LONGER_ITERS}_updates')
    comparison = pd.concat([pilot_table.loc[pilot_table.method == 'frengression_joint'],
                            longer_table], ignore_index=True)
    display(comparison[['method', 'arm', 'estimated_mean1', 'estimated_mean2',
                        'mean_mae', 'estimated_var1', 'estimated_var2',
                        'estimated_cov12', 'cov12_error']].round(4))
    print(f'Longer fit time: {longer_seconds:.1f} seconds')
    if (comparison.loc[comparison.method == f'frengression_{LONGER_ITERS}_updates',
                      'mean_mae'].max() > 0.2 or longer_table.cov12_error.max() > 0.2):
        print('The longer fit has substantial moment errors; inspect this training budget before using it.')
    else:
        print('The longer fit passes this coarse moment check; inspect calibration and repeat the selected budget.')
else:
    print('Extra 4000-update fit skipped; overnight replications use 1000 updates.')

# The same selected training budget is used in the report and every replication.
if PAPER_FIT == 'pilot':
    paper_model, paper_draws, paper_fit_seconds = joint_model, joint_draws, fit_seconds
    PAPER_ITERS = FR_ITERS
elif PAPER_FIT == 'longer':
    paper_model, paper_draws, paper_fit_seconds = longer_model, longer_draws, longer_seconds
    PAPER_ITERS = LONGER_ITERS
else:
    raise ValueError("PAPER_FIT must be 'pilot' or 'longer'")
print(f'Reporting fit: {PAPER_FIT}, {PAPER_ITERS} updates')

### Sampling and Gaussian-noise checks

Compare direct causal samples with the fitted composition `f(a, h(Z,a,xi))`. The auxiliary noise target is `N(0,I_2)`; report its means, covariance and joint CDF errors.


In [ ]:
paper_model.eval()
torch.manual_seed(SEED + 501)
rng = np.random.default_rng(SEED + 502)
diagnostic_rows = []
latent_draws = {}
implied_draws = {}
with torch.no_grad():
    xt = torch.as_tensor(x, dtype=torch.float32)
    zt = torch.as_tensor(z, dtype=torch.float32)
    eta_observed = paper_model.model_eta(torch.cat([xt, zt], dim=1))
    y_observed_fit = paper_model.model_y(torch.cat([xt, eta_observed], dim=1)).cpu().numpy()
    for arm in (0, 1):
        mask = x[:, 0] == arm
        x_ref = torch.full((MC_DRAWS, 1), float(arm))
        eta_ref = torch.randn(MC_DRAWS, 2)
        explicit = paper_model.model_y(torch.cat([x_ref, eta_ref], dim=1)).cpu().numpy()
        api = paper_draws[arm]
        # Each z is marginal, independently of the fixed intervention arm.
        z_ref = torch.as_tensor(z[rng.permutation(len(z))], dtype=torch.float32)
        x_latent = torch.full((len(z), 1), float(arm))
        eta_refit = paper_model.model_eta(torch.cat([x_latent, z_ref], dim=1)).cpu().numpy()
        latent_draws[arm] = eta_refit
        implied_draws[arm] = paper_model.model_y(torch.cat([x_latent,
            torch.as_tensor(eta_refit, dtype=torch.float32)], dim=1)).cpu().numpy()
        for label, values in [('causal_via_f_h_empirical_Z', implied_draws[arm]),
                              ('causal_api', api), ('causal_explicit_eta', explicit),
                              ('eta_given_arm', eta_refit),
                              ('observed_data', y[mask]),
                              ('observed_fit', y_observed_fit[mask])]:
            mu = values.mean(axis=0)
            cov = np.cov(values, rowvar=False)
            diagnostic_rows.append(dict(arm=arm, source=label, n=len(values),
                                        mean1=mu[0], mean2=mu[1],
                                        var1=cov[0, 0], var2=cov[1, 1],
                                        cov12=cov[0, 1]))
diagnostic_table = pd.DataFrame(diagnostic_rows)
display(diagnostic_table.round(3))
print('eta_given_arm target: means (0, 0), variances (1, 1), covariance 0.')
print('causal_api and causal_explicit_eta should agree within simulation error.')
print('observed_data versus observed_fit checks model fit, not causal recovery.')

## Joint probabilities

In addition to means, covariance and marginal quantiles, evaluate the joint CDF on a fixed 25-point grid and the events `Y1>0, Y2>0` and `Y1>1, Y2>-0.5`. Treatment contrasts use the same thresholds in both arms.

The probability that both outcomes exceed their true arm medians is a dependence check: its truth is approximately `0.3155`, versus `0.25` for independent exact margins. Reference probabilities use numerical integration of the specified Gaussian law. Model-draw MCSE describes sampling error for a fitted model.


In [ ]:
QUANTILE_LEVELS = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
CDF_GRID_OFFSETS = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
FIXED_EVENTS = {'both_above_zero': (0.0, 0.0),
                'Y1_above_1_Y2_above_minus_half': (1.0, -0.5)}

def validate_draws(values):
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[1] != 2 or len(values) < 2:
        raise ValueError(f'Expected at least two [draws, 2] joint draws, got {values.shape}')
    if not np.isfinite(values).all():
        raise ValueError('Non-finite joint draws')
    return values

@lru_cache(maxsize=4096)
def standard_normal_joint_cdf(t1, t2, rho):
    if not -1.0 < rho < 1.0:
        raise ValueError('A nonsingular Gaussian correlation is required')
    if abs(rho) < 1e-14:
        return float(norm.cdf(t1) * norm.cdf(t2))
    value, error = quad(lambda u: norm.pdf(u) * norm.cdf(
        (t2 - rho * u) / np.sqrt(1.0 - rho**2)),
        -np.inf, t1, epsabs=1e-10, epsrel=1e-10)
    if error > 1e-7 or not np.isfinite(value):
        raise RuntimeError('Gaussian joint-probability integration failed')
    return float(value)

def gaussian_joint_probability(thresholds, mean, covariance, upper=False):
    mean, covariance = np.asarray(mean), np.asarray(covariance)
    sd = np.sqrt(np.diag(covariance))
    if np.any(sd <= 0) or not np.isfinite(covariance).all():
        raise ValueError('Invalid Gaussian covariance')
    t1, t2 = (np.asarray(thresholds) - mean) / sd
    rho = float(covariance[0, 1] / np.prod(sd))
    # Central symmetry avoids subtracting nearly equal tail probabilities.
    if upper:
        t1, t2 = -t1, -t2
    return standard_normal_joint_cdf(float(t1), float(t2), rho)

def cdf_grid_errors(values, mean, covariance):
    d = validate_draws(values)
    sd = np.sqrt(np.diag(covariance))
    thresholds = np.asarray(mean) + CDF_GRID_OFFSETS[:, None] * sd
    errors = []
    for t1 in thresholds[:, 0]:
        for t2 in thresholds[:, 1]:
            estimate = np.mean((d[:, 0] <= t1) & (d[:, 1] <= t2))
            truth = gaussian_joint_probability((t1, t2), mean, covariance)
            errors.append(abs(estimate - truth))
    return float(np.mean(errors)), float(np.max(errors))

def evaluate_events(draws, label):
    rows = []
    for arm in (0, 1):
        d = validate_draws(draws[arm])
        events = {**FIXED_EVENTS, 'both_above_true_arm_medians': tuple(TRUE_MEAN[arm])}
        for event, thresholds in events.items():
            t1, t2 = thresholds
            estimate = float(np.mean((d[:, 0] > t1) & (d[:, 1] > t2)))
            truth = gaussian_joint_probability(thresholds, TRUE_MEAN[arm], TRUE_COV, upper=True)
            fitted_product = float(np.mean(d[:, 0] > t1) * np.mean(d[:, 1] > t2))
            true_product = gaussian_joint_probability(thresholds, TRUE_MEAN[arm],
                                                      np.diag(np.diag(TRUE_COV)), upper=True)
            rows.append(dict(method=label, arm=arm, event=event, threshold1=t1, threshold2=t2,
                fixed_across_arms=event in FIXED_EVENTS, true_probability=truth,
                estimated_probability=estimate, probability_error=abs(estimate - truth),
                signed_probability_error=estimate - truth,
                true_independent_probability=true_product,
                fitted_independent_probability=fitted_product,
                true_dependence_gap=truth - true_product,
                estimated_dependence_gap=estimate - fitted_product,
                model_draw_mcse=np.sqrt(estimate * (1.0 - estimate) / len(d)), n_draws=len(d)))
    return pd.DataFrame(rows)

def evaluate_joint_full(draws, label):
    moments = evaluate_joint(draws, label)
    rows = []
    events = evaluate_events(draws, label)
    for arm in (0, 1):
        d = validate_draws(draws[arm])
        grid_mae, grid_max = cdf_grid_errors(d, TRUE_MEAN[arm], TRUE_COV)
        true_quantiles = TRUE_MEAN[arm] + norm.ppf(QUANTILE_LEVELS)[:, None] * np.sqrt(np.diag(TRUE_COV))
        rows.append(dict(method=label, arm=arm,
            marginal_quantile_mae=float(np.abs(np.quantile(d, QUANTILE_LEVELS, axis=0) - true_quantiles).mean()),
            joint_cdf_grid_mae=grid_mae, joint_cdf_grid_max_error=grid_max,
            joint_event_mae=float(events.loc[events.arm == arm, 'probability_error'].mean()),
            n_draws=len(d)))
    return moments.merge(pd.DataFrame(rows), on=['method', 'arm'], validate='one_to_one')

def fixed_event_contrasts(events):
    rows = []
    selected = events.loc[events.fixed_across_arms]
    group_keys = ['method', 'event']
    if 'seed' in selected.columns:
        group_keys.insert(0, 'seed')
    for key, group in selected.groupby(group_keys, sort=False):
        arm_rows = group.set_index('arm')
        if set(arm_rows.index) != {0, 1} or len(arm_rows) != 2:
            continue  # A failed or missing arm is not a valid treatment contrast.
        values = key if isinstance(key, tuple) else (key,)
        row = dict(zip(group_keys, values))
        truth = arm_rows.loc[1, 'true_probability'] - arm_rows.loc[0, 'true_probability']
        estimate = arm_rows.loc[1, 'estimated_probability'] - arm_rows.loc[0, 'estimated_probability']
        row.update(true_contrast=truth, estimated_contrast=estimate,
                   contrast_error=abs(estimate - truth), signed_contrast_error=estimate - truth,
                   model_draw_contrast_mcse=float(np.sqrt(np.square(arm_rows.model_draw_mcse).sum())))
        rows.append(row)
    return pd.DataFrame(rows)

def shuffle_second_coordinate(draws, seed):
    rng = np.random.default_rng(seed)
    return {arm: np.column_stack((validate_draws(draws[arm])[:, 0],
                  rng.permutation(validate_draws(draws[arm])[:, 1]))) for arm in (0, 1)}

## Conditional engression plus g-computation

Fit one joint model of `(Y1,Y2) | A,Z`, fix treatment, then draw covariates from their empirical marginal distribution. Both outcomes are sampled together. Network settings and update counts match those used for Frengression; Frengression fits two networks (`f` and `h`).


In [ ]:
def fit_conditional_gcomp(x, z, y, seed, iterations, draws=MC_DRAWS):
    torch.manual_seed(seed)
    device = torch.device('cpu')
    model = StoNet(x.shape[1] + z.shape[1], y.shape[1],
                   3, 100, 2, add_bn=False, noise_all_layer=False).to(device)
    inputs = torch.as_tensor(np.column_stack((x, z)), dtype=torch.float32, device=device)
    targets = torch.as_tensor(y, dtype=torch.float32, device=device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    model.train()
    start = time.perf_counter()
    for iteration in range(iterations):
        optimizer.zero_grad()
        sample1, sample2 = model(inputs), model(inputs)
        loss, _, _ = energy_loss_two_sample(targets, sample1, sample2)
        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite conditional-engression loss')
        loss.backward()
        optimizer.step()
        if iteration == 0 or (iteration + 1) % max(1, iterations // 5) == 0:
            print(f'Conditional g-computation update {iteration + 1}: loss {loss.item():.4f}')
    seconds = time.perf_counter() - start
    model.eval()
    rng = np.random.default_rng(seed + 700)
    arm_draws = {}
    with torch.no_grad():
        for arm in (0, 1):
            sampled_z = z[rng.integers(0, len(z), size=draws)]
            reference = np.column_stack((np.full((draws, x.shape[1]), arm), sampled_z))
            outputs = [model(torch.as_tensor(batch, dtype=torch.float32, device=device)).cpu().numpy()
                       for batch in np.array_split(reference, max(1, (draws + 2047) // 2048))]
            arm_draws[arm] = validate_draws(np.concatenate(outputs, axis=0))
    return model, arm_draws, seconds

RUN_GCOMP_PILOT = True
GCOMP_ITERS = PAPER_ITERS
paper_methods = {'frengression_joint': paper_draws,
                 'shuffled_diagnostic': shuffle_second_coordinate(paper_draws, SEED + 99),
                 'frengression_gcomp_audit': implied_draws}
paper_times = {'frengression_joint': paper_fit_seconds,
               'shuffled_diagnostic': np.nan, 'frengression_gcomp_audit': np.nan}
PILOT_FAILURES = []
if RUN_GCOMP_PILOT:
    try:
        gcomp_model, gcomp_draws, gcomp_seconds = fit_conditional_gcomp(
            x, z, y, SEED, iterations=GCOMP_ITERS, draws=MC_DRAWS)
        paper_methods['conditional_engression_gcomp'] = gcomp_draws
        paper_times['conditional_engression_gcomp'] = gcomp_seconds
    except Exception as error:
        PILOT_FAILURES.append(dict(seed=SEED, method='conditional_engression_gcomp', error=str(error)))
        print('Pilot comparison failed; recorded and retried during replications:', error)


In [ ]:
paper_table = pd.concat([evaluate_joint_full(draws, method)
                        for method, draws in paper_methods.items()], ignore_index=True)
paper_table['fit_seconds'] = paper_table.method.map(paper_times)
event_table = pd.concat([evaluate_events(draws, method)
                        for method, draws in paper_methods.items()], ignore_index=True)
contrast_table = fixed_event_contrasts(event_table)
latent_rows = []
for arm in (0, 1):
    values = validate_draws(latent_draws[arm])
    mu, covariance = values.mean(axis=0), np.cov(values, rowvar=False)
    grid_mae, grid_max = cdf_grid_errors(values, np.zeros(2), np.eye(2))
    latent_rows.append(dict(arm=arm, mean1=mu[0], mean2=mu[1],
        var1=covariance[0, 0], var2=covariance[1, 1], cov12=covariance[0, 1],
        normal_cdf_grid_mae=grid_mae, normal_cdf_grid_max_error=grid_max))
latent_table = pd.DataFrame(latent_rows)

print('Moments: truth is (0,0) or (2,-1), unit variances and covariance 0.4')
display(paper_table.set_index(['method', 'arm'])[['estimated_mean1', 'estimated_mean2',
    'mean_mae', 'estimated_var1', 'estimated_var2', 'estimated_cov12',
    'cov12_error', 'covariance_fro_error']].round(4))
print('Marginal quantiles and joint probabilities on a prespecified grid')
display(paper_table.set_index(['method', 'arm'])[['marginal_quantile_mae',
    'joint_cdf_grid_mae', 'joint_cdf_grid_max_error', 'joint_event_mae', 'fit_seconds']].round(4))
print('Joint event probabilities; model-draw MCSE excludes model-estimation uncertainty')
display(event_table.set_index(['method', 'arm', 'event'])[['true_probability',
    'estimated_probability', 'probability_error', 'true_independent_probability',
    'true_dependence_gap', 'estimated_dependence_gap', 'model_draw_mcse']].round(4))
print('Treatment contrasts for the two events with fixed thresholds')
display(contrast_table.set_index(['method', 'event']).round(4))
print('Latent calibration under the empirical Z law: target N(0,I_2)')
display(latent_table.set_index('arm').round(4))


In [ ]:
RESULT_DIR = Path.home() / 'frengression_results' / 'joint_outcomes_causl'
PILOT_OUTPUT = RESULT_DIR / f'pilot_seed{SEED}_fr{PAPER_ITERS}_mc{MC_DRAWS}'
SAVE_PILOT = True

def installed_version(name):
    try:
        return package_metadata.version(name)
    except package_metadata.PackageNotFoundError:
        return None

git_info = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'],
                          capture_output=True, text=True)
run_metadata = dict(n=N, seed=SEED, instrument_strength=STRENGTH_INSTR,
    paper_fit=PAPER_FIT, frengression_updates=PAPER_ITERS, gcomp_updates=GCOMP_ITERS,
    gcomp_pilot_enabled=RUN_GCOMP_PILOT, mc_draws=MC_DRAWS, noise_dim=2,
    layers=3, width=100, learning_rate=1e-4,
    true_mean={str(k): v.tolist() for k, v in TRUE_MEAN.items()},
    true_covariance=TRUE_COV.tolist(), fixed_events=FIXED_EVENTS,
    quantile_levels=QUANTILE_LEVELS.tolist(), cdf_grid_offsets=CDF_GRID_OFFSETS.tolist(),
    repository_commit=git_info.stdout.strip() if git_info.returncode == 0 else None,
    python=platform.python_version(), platform=platform.platform(),
    model_source_sha256=hashlib.sha256((ROOT / 'src/frengression/frengression.py').read_bytes()).hexdigest(),
    versions={name: installed_version(name) for name in
              ['torch', 'engression', 'numpy', 'pandas', 'scipy']},
    uncertainty_note='model_draw_mcse is Monte Carlo variability conditional on the fitted model',
    timing_note='outcome-module fitting only; excludes R generation and complete simulator fitting')
if SAVE_PILOT:
    PILOT_OUTPUT.mkdir(parents=True, exist_ok=True)
    for name, table in [('moments_and_joint_scores', paper_table), ('joint_events', event_table),
                        ('fixed_event_contrasts', contrast_table), ('noise_checks', latent_table),
                        ('sampling_audit', diagnostic_table)]:
        table.to_csv(PILOT_OUTPUT / f'{name}.csv', index=False)
    (PILOT_OUTPUT / 'metadata.json').write_text(json.dumps(run_metadata, indent=2))
    print('Saved pilot report:', PILOT_OUTPUT)
    pd.DataFrame(PILOT_FAILURES, columns=['seed', 'method', 'error']).to_csv(
        PILOT_OUTPUT / 'failures.csv', index=False)


### Read the figure

Gray points are generated outcome pairs. Dashed black curves show the true Gaussian 50% and 95% regions; solid blue ellipses use the generated sample's mean and covariance. Closer overlap indicates better agreement in these moments. The displayed covariance error is the Frobenius norm.

Use the 30-replication tables and paired-error plot for method comparisons. Both treatment arms have true covariance `0.4`.


In [ ]:
fig = plot_joint_distributions(paper_methods, TRUE_MEAN, TRUE_COV,
    PILOT_OUTPUT / 'joint_distributions' if SAVE_PILOT else None)
plt.show()


## Paired replications

Run 30 datasets with the fixed 1000-update budget. Both fitted methods see the same data; completed fits are reused from checkpoints. Failure counts remain in the report.

Summary tables use all completed runs. The distribution figure uses the first complete pair in the prespecified order, without selecting on accuracy. The paired-error figure shows covariance and joint CDF errors across datasets.


In [ ]:
RUN_REPEATS = True
N_REPEATS = 30
RUN_GCOMP_REPEATS = True
SUMMARY_METRICS = ['mean_mae', 'cov12_error', 'covariance_fro_error',
                   'marginal_quantile_mae', 'joint_cdf_grid_mae', 'joint_event_mae']

def summarize_errors(table, group_keys, metrics):
    rows = []
    for key, group in table.groupby(group_keys, sort=False):
        values = key if isinstance(key, tuple) else (key,)
        for metric in metrics:
            scores = group[metric].dropna().to_numpy(dtype=float)
            if not len(scores):
                continue
            row = dict(zip(group_keys, values))
            sd = float(np.std(scores, ddof=1)) if len(scores) > 1 else np.nan
            row.update(metric=metric, n=len(scores), mean=float(scores.mean()),
                       sd=sd, mcse_of_mean=sd / np.sqrt(len(scores)))
            rows.append(row)
    return pd.DataFrame(rows)

def paired_error_summary(table, comparator, metrics=SUMMARY_METRICS):
    fr = table.loc[table.method == 'frengression_joint'].set_index(['seed', 'arm'])
    other = table.loc[table.method == comparator].set_index(['seed', 'arm'])
    if not fr.index.is_unique or not other.index.is_unique:
        raise ValueError('Repeated method/seed/arm scores')
    common = fr.index.intersection(other.index)
    if not len(common):
        return pd.DataFrame()
    differences = (other.loc[common, metrics] - fr.loc[common, metrics]).reset_index()
    summary = summarize_errors(differences, ['arm'], metrics)
    summary.insert(0, 'comparator', comparator)
    return summary.rename(columns={'n': 'n_pairs', 'mean': 'mean_error_difference'})


def draw_noise_checks(model, covariates, seed):
    model.eval()
    torch.manual_seed(seed + 801)
    extras = {}
    with torch.no_grad():
        zz = torch.as_tensor(covariates, dtype=torch.float32)
        for arm in (0, 1):
            aa = torch.full((len(zz), 1), float(arm))
            eta = model.model_eta(torch.cat([aa, zz], dim=1))
            implied = model.model_y(torch.cat([aa, eta], dim=1))
            extras[f'eta{arm}'] = eta.cpu().numpy()
            extras[f'implied{arm}'] = implied.cpu().numpy()
    return extras


if RUN_REPEATS:
    repeats_dir = RESULT_DIR / (f'repeats_v2_n{N}_s{STRENGTH_INSTR:g}_fr{PAPER_ITERS}_'
                                f'gcomp{GCOMP_ITERS}_mc{MC_DRAWS}_k{N_REPEATS}')
    repeats_dir.mkdir(parents=True, exist_ok=True)
    repeat_metadata = {**run_metadata, 'n_repeats': N_REPEATS,
                       'gcomp_repeats_enabled': RUN_GCOMP_REPEATS,
                       'seeds': list(range(SEED, SEED + N_REPEATS)), 'checkpoint_format': 2}
    (repeats_dir / 'metadata.json').write_text(json.dumps(repeat_metadata, indent=2))
    all_scores, all_events, all_noise, failures, completion_rows = [], [], [], [], []
    methods_requested = ['frengression_joint']
    if RUN_GCOMP_REPEATS:
        methods_requested.append('conditional_engression_gcomp')
    representative = None
    for seed in range(SEED, SEED + N_REPEATS):
        dataset, seed_methods, seed_extras = None, {}, {}
        for method in methods_requested:
            protocol = dict(format=2, n=N, seed=seed, strength=STRENGTH_INSTR,
                method=method, fr_iters=PAPER_ITERS, gcomp_iters=GCOMP_ITERS,
                mc_draws=MC_DRAWS, layers=3, width=100, noise_dim=2, learning_rate=1e-4,
                model_source_sha256=run_metadata['model_source_sha256'])
            checkpoint = repeats_dir / f'seed{seed}_{method}.npz'
            try:
                cached = load_draw_checkpoint(checkpoint, protocol)
                if cached is None:
                    if dataset is None:
                        dataset = generate_causl_joint(N, seed, STRENGTH_INSTR)
                    xx, zz, yy, _ = dataset
                    print(f'Fitting {method}: seed {seed}', flush=True)
                    if method == 'frengression_joint':
                        model, draws, seconds = fit_joint(xx, zz, yy, seed,
                            iterations=PAPER_ITERS, draws=MC_DRAWS)
                        extras = draw_noise_checks(model, zz, seed)
                    else:
                        model, draws, seconds = fit_conditional_gcomp(xx, zz, yy, seed,
                            iterations=GCOMP_ITERS, draws=MC_DRAWS)
                        extras = {}
                    save_draw_checkpoint(checkpoint, protocol, draws, seconds, extras)
                    del model
                else:
                    draws, seconds, extras = cached
                    print(f'Reusing {method}: seed {seed}', flush=True)
                scores, events = evaluate_joint_full(draws, method), evaluate_events(draws, method)
                scores.insert(0, 'seed', seed); events.insert(0, 'seed', seed)
                scores['fit_seconds'] = seconds
                all_scores.append(scores); all_events.append(events)
                seed_methods[method], seed_extras[method] = draws, extras
                atomic_csv(pd.concat(all_scores, ignore_index=True), repeats_dir / 'scores.csv')
                atomic_csv(pd.concat(all_events, ignore_index=True), repeats_dir / 'events.csv')
            except Exception as error:
                failures.append(dict(seed=seed, method=method, error=str(error)))
                print(f'Failed {method}, seed={seed}: {error}', flush=True)
                atomic_csv(pd.DataFrame(failures, columns=['seed', 'method', 'error']),
                           repeats_dir / 'failures.csv')
        if 'frengression_joint' in seed_methods:
            extras = seed_extras['frengression_joint']
            shuffled = shuffle_second_coordinate(seed_methods['frengression_joint'], seed + 99)
            diagnostic_methods = {'shuffled_diagnostic': shuffled}
            if all(name in extras for name in ('eta0', 'eta1', 'implied0', 'implied1')):
                diagnostic_methods['frengression_gcomp_audit'] = {a: extras[f'implied{a}'] for a in (0, 1)}
                for arm in (0, 1):
                    values = validate_draws(extras[f'eta{arm}'])
                    covariance = np.cov(values, rowvar=False)
                    grid_mae, grid_max = cdf_grid_errors(values, np.zeros(2), np.eye(2))
                    all_noise.append(dict(seed=seed, arm=arm, n=len(values),
                        mean1=values[:, 0].mean(), mean2=values[:, 1].mean(),
                        var1=covariance[0, 0], var2=covariance[1, 1], cov12=covariance[0, 1],
                        normal_cdf_grid_mae=grid_mae, normal_cdf_grid_max_error=grid_max))
            for method, draws in diagnostic_methods.items():
                scores, events = evaluate_joint_full(draws, method), evaluate_events(draws, method)
                scores.insert(0, 'seed', seed); events.insert(0, 'seed', seed)
                scores['fit_seconds'] = np.nan
                all_scores.append(scores); all_events.append(events)
            if representative is None and set(methods_requested).issubset(seed_methods):
                representative = (seed, {**seed_methods, 'shuffled_diagnostic': shuffled})
        completion_rows.append(dict(seed=seed,
            frengression_complete='frengression_joint' in seed_methods,
            comparator_complete='conditional_engression_gcomp' in seed_methods,
            complete_pair=set(methods_requested).issubset(seed_methods)))
        if all_scores:
            atomic_csv(pd.concat(all_scores, ignore_index=True), repeats_dir / 'scores.csv')
            atomic_csv(pd.concat(all_events, ignore_index=True), repeats_dir / 'events.csv')
        atomic_csv(pd.DataFrame(all_noise), repeats_dir / 'noise_checks.csv')
        atomic_csv(pd.DataFrame(completion_rows), repeats_dir / 'completion.csv')
        atomic_csv(pd.DataFrame(failures, columns=['seed', 'method', 'error']), repeats_dir / 'failures.csv')
    if not all_scores:
        raise RuntimeError(f'No completed fits; inspect {repeats_dir / "failures.csv"}')
    repeated_results = pd.concat(all_scores, ignore_index=True)
    repeated_events = pd.concat(all_events, ignore_index=True)
    repeated_contrasts = fixed_event_contrasts(repeated_events)
    score_summary = summarize_errors(repeated_results, ['method', 'arm'], SUMMARY_METRICS)
    event_summary = summarize_errors(repeated_events, ['method', 'arm', 'event'], ['probability_error'])
    contrast_summary = summarize_errors(repeated_contrasts, ['method', 'event'],
                                        ['contrast_error', 'signed_contrast_error'])
    paired_tables = [paired_error_summary(repeated_results, comparator)
                     for comparator in ['shuffled_diagnostic', 'conditional_engression_gcomp']]
    paired_tables = [table for table in paired_tables if not table.empty]
    paired_summary = pd.concat(paired_tables, ignore_index=True) if paired_tables else pd.DataFrame()
    noise_summary = summarize_errors(pd.DataFrame(all_noise), ['arm'],
        ['mean1', 'mean2', 'var1', 'var2', 'cov12', 'normal_cdf_grid_mae']) if all_noise else pd.DataFrame()
    for name, table in [('score_summary', score_summary), ('event_summary', event_summary),
                        ('contrast_summary', contrast_summary), ('paired_error_summary', paired_summary),
                        ('fixed_event_contrasts', repeated_contrasts), ('noise_summary', noise_summary)]:
        atomic_csv(table, repeats_dir / f'{name}.csv')
    print(f'Planned datasets: {N_REPEATS}; complete pairs: '
          f'{sum(row["complete_pair"] for row in completion_rows)}; failed attempts: {len(failures)}')
    display(pd.DataFrame(failures, columns=['seed', 'method', 'error']))
    display(score_summary.round(4)); display(event_summary.round(4))
    display(contrast_summary.round(4)); display(paired_summary.round(4))
    if representative is not None:
        representative_seed, representative_methods = representative
        plot_joint_distributions(representative_methods, TRUE_MEAN, TRUE_COV,
            repeats_dir / 'joint_distributions')
        plt.show()
    comparison_figure = plot_paired_errors(repeated_results, repeats_dir / 'paired_error_comparison')
    if comparison_figure is not None:
        plt.show()
    print('Saved full joint-outcome report:', repeats_dir)
